# Miniproject template

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="miniproject")

## Generic scene

### Available entities

The scene contains **12 agents** (blue squares) and **32 objects** (green circles). You can freely customize these 44 entities in order to implement you own scenario.

### Available subtypes

The scene provides **8 generic subtypes**. The names of these subtypes can be accessed with

In [ ]:
controller.subtypes

### Assigning subtypes and attributes to entities

As an example, let's consider the following scenario:

- Among the 12 agents:
    - 8 are considered as *prey* agents. They are assigned with the subtype `"agent_subtype_1", a diameter of 4, the color blue, and a maximum speed of 2.
    - 4 are considered as *predator* agents. They are assigned with the subtype `"agent_subtype_2", a diameter of 6, the color red, and a maximum speed of 1.
- Among the 32 object
    -  24 are considered as *resources*. They are assigned with the subtype `"object_subtype_1", a diameter of 3 and the color green.
    -  8 are consided ad *obstacles¨. They are assigned with the subtype `"object_subtype_1", a diameter of 8 and the color orange.
 
To set up the scene outlines above we can write:

In [ ]:
# Defining prey agents
for agent in controller.agents[0:8]:
    agent.subtype = "agent_subtype_1"
    agent.diameter = 4
    agent.color = "blue"
    agent.max_speed = 2


# Defining predator agents
for agent in controller.agents[8:12]:
    agent.subtype = "agent_subtype_2"
    agent.diameter = 6
    agent.color = "red"
    agent.max_speed = 1


# Defining resource objects
for obj in controller.objects[0:24]:
    obj.subtype = "object_subtype_1"
    obj.diameter = 3
    obj.color = "green"


# Defining obstacle objects
for obj in controller.objects[24:32]:
    obj.subtype = "object_subtype_2"
    obj.diameter = 8
    obj.color = "orange"

### Attaching behaviors to specific agent's subtypes

In [ ]:
def obstacle_avoidance(agent):
    left, right = agent.proximeters(sensed_entities=["object_subtype_2"])
    left_motor = 1 - right
    right_motor = 1 - left
    return left_motor, right_motor  


def foraging(agent):
    left, right = agent.proximeters(sensed_entities=["object_subtype_1"])
    left_motor = right
    right_motor = left
    return left_motor, right_motor


def attack(agent):
    left, right = agent.proximeters(sensed_entities=["agent_subtype_1"])
    left_motor = right
    right_motor = left
    return left_motor, right_motor


def fear(agent):
    left, right = agent.proximeters(sensed_entities=["agent_subtype_2"])
    left_motor = left
    right_motor = right
    return left_motor, right_motor

In [ ]:
# Detach potential previous behaviors from all agents
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)


# Attach behaviors to prey agents
for agent in controller.agents:
    if agent.subtype == "agent_subtype_1":
        agent.attach_behavior(obstacle_avoidance)
        agent.attach_behavior(foraging)
        agent.attach_behavior(fear)


# Attach behaviors to predator agents
for agent in controller.agents:
    if agent.subtype == "agent_subtype_2":
        agent.attach_behavior(obstacle_avoidance)
        agent.attach_behavior(attack)

### Launching mutliple consumption mechanisms

The scene provides **4 independent *slots* for the consumption mechanim**. Below we use to of them, `slot_1` and `slot_2` as an example where:

- Preys consume resources
- Predators consume preys

In [ ]:
# Preys consume resources
controller.consumption.slot_1.source_subtype = "agent_subtype_1"
controller.consumption.slot_1.target_subtype = "object_subtype_1"
controller.consumption.slot_1.start = True

In [ ]:
# Predators consume preys
controller.consumption.slot_2.source_subtype = "agent_subtype_2"
controller.consumption.slot_2.target_subtype = "agent_subtype_1"
controller.consumption.slot_2.start = True

### Launching mutliple spawning mechanisms

The scene provides **4 independent *slots* for the spawning mechanim**. Below we use to of them, `slot_1` and `slot_2` as an example where:

- New resources spawn every 100 time steps
- New preys spawn every 200 time steps

In [ ]:
# Spawn resources
controller.spawn.slot_1.subtype = "object_subtype_1"
controller.spawn.slot_1.period = 100
controller.spawn.slot_1.start = True

In [ ]:
# Spawn preys
controller.spawn.slot_2.subtype = "agent_subtype_1"
controller.spawn.slot_2.period = 200
controller.spawn.slot_2.start = True

## Avalaible entity attributes

You can freely customize the attributes of the entities (either agents or objects).

### All entities

x_position, y_position, orientation, subtype, diameter, color, friction, mass, exists

### Agents
proxs_dist_max, proxs_cos_min, max_speed, wheel_diameter, visible_wheels, visible_proxs, visible

Note: friction seems to no longer make agent drift (probably to change in motor force, which is now always aligned with the front direction..). But one advantage is that reducing the friction makes them move much faster ..

Note (solved): Current it seems that agents can't consume other agents. Might just be a matter of adding an reproduction component to agents in the config (for their death).
**BUT:**
Does spawning of agent deal correctly with subtype? (as it chooses a random non-existing agent of the list, it might be of the wrong subtype. And also wrong behavior, physical_attributes etc..). This might be hard to solve, maube better indicate that agent spawning is not available, creativity emerges from constrains anyway .. (or just explain the limitation). Might be solved with routines, but it's a workaround. And requires controller routines. 
**ACTUALLY**
Yes, only entities of the indicated spawning subtype do spawn, so normally all good.

## Calibrating the simulator speed

This session's environment contains more entities than in the previous sessions, which might slow down the simulation. Let's attach the `obstacle_avoidance` behavior to all agents so that you can observe if it runs fast enough:

In [ ]:
for agent in controller.agents:
    agent.attach_behavior(obstacle_avoidance)

In case you find the agents are moving too slow, you can increase the number of steps the simulation performs on the server for each step performed in this notebook's controller by mofifying the `controller.simulator.env.num_scan_steps` parameter. It is set to 1 by default. If you double it to 2, your simulation will run approximately twice faster:

In [ ]:
controller.simulator.env.num_scan_steps = 2

Use this mechanism wisely, as increasing this number too high will make your agent's behaviors less reactive, in the sense that the time between the proximeter sensing and the motor activations will be longer. We recommend to not increase it above 8 maximum. Only use integer numbers for this parameter. Once you have find a number that suits you, you can detach the obstacle_avoidance behavior with:

In [ ]:
for agent in controller.agents:
    agent.detach_behavior(obstacle_avoidance, stop_motors=True)